In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Đường dẫn đến thư mục dữ liệu thô (tự động nhận diện local hoặc Google Colab)
RAW = Path("RAW_DATA/raw") if Path("RAW_DATA/raw").exists() else Path("/content/drive/MyDrive/[LTPTDL] Đồ án/raw")

FILES = [
    "Fact", "Calendar", "Business Line", "Group", "Subgroup",
    "Channel", "Region", "Headlines", "KPI", "Ratios"
]

# Đọc toàn bộ bảng dữ liệu dưới dạng chuỗi
tables = {
    f: pd.read_csv(RAW / f"{f}.csv", dtype=str, encoding="utf-8-sig")
    for f in FILES
}

fact = tables["Fact"]

# Kiểm tra kích thước các bảng
{name: df.shape for name, df in tables.items()}

In [ ]:
# Chuẩn bị các chỉ số chẩn đoán từ quá trình khám phá dữ liệu đơn biến và đa biến
eda = fact.copy()

def parse_amount(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    neg = s.startswith("(") or s.startswith("-")
    s = re.sub(r"USD|\$|[()\s-]", "", s, flags=re.I)
    if re.fullmatch(r"\d{1,3}(?:\.\d{3})+,\d{2}|\d+,\d{2}", s):
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", "")
    return (-1 if neg else 1) * float(s)

eda["Amount_num"] = eda["Amount, $"].map(parse_amount)

# 1. Khóa liên kết không hợp lệ
keys = {
    "Business Line ID": "Business Line",
    "Group ID": "Group",
    "Subgroup ID": "Subgroup"
}
invalid_keys = pd.Series({
    col: (~fact[col].isin(tables[table][col]) & fact[col].notna()).sum()
    for col, table in keys.items()
}, name="Invalid_Count")

# 2. Phát hiện Outlier
sg_name = tables["Subgroup"].set_index("Subgroup ID")["Subgroup"]
eda["SG"] = eda["Subgroup ID"].map(sg_name)
d = eda.dropna(subset=["SG", "Amount_num"]).copy()
d = d[d["Amount_num"] != 0]
d["Log_Abs"] = np.log10(d["Amount_num"].abs())

def iqr_flag(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

d["Outlier"] = d.groupby("SG")["Log_Abs"].transform(iqr_flag)

# 3. Chuẩn hóa tạm thời để kiểm tra tính nhất quán giữa các cột
eda["RevExp_c"] = eda["Revenue or expense"].str.strip().str.lower().map({
    "revenue": "Revenue", "rev": "Revenue", "expense": "Expense", "expenses": "Expense", "exp": "Expense"
})
sg2g = tables["Subgroup"].set_index("Subgroup ID")["Group ID"]
eda["Group_SG"] = eda["Subgroup ID"].map(sg2g)

ok = eda["Group_SG"].notna()
grp_bad = ok & (eda["Group ID"] != eda["Group_SG"])
exp_label = np.where(eda["Group_SG"] == "1", "Revenue", "Expense")
re_bad = ok & (eda["RevExp_c"] != exp_label)
valid_amt = ok & eda["Amount_num"].notna() & (eda["Amount_num"] != 0)
exp_sign = np.where(eda["Group_SG"] == "1", 1, -1)
sign_bad = valid_amt & (np.sign(eda["Amount_num"]) != exp_sign)

bl_tab = tables["Business Line"].copy()
bl_tab["clean"] = bl_tab["Business Line"].str.replace("\xa0", " ", regex=False).str.strip()
name2id = bl_tab.set_index("clean")["Business Line ID"]
exp_id = eda["Business Line Name"].map(name2id)
bl_valid = eda["Business Line ID"].isin(bl_tab["Business Line ID"])
bl_bad = bl_valid & (eda["Business Line ID"] != exp_id)

print("Đã hoàn tất tính toán các chỉ số chẩn đoán chuẩn bị lập Bảng chẩn đoán tổng hợp.")

## 1.3. Bảng chẩn đoán tổng hợp


In [ ]:
n = len(fact)

# Số ô sai định dạng: Date không đúng YYYY-MM-DD; Amount không phải số thông thường
n_date_fmt = (fact["Date"].notna() & pd.to_datetime(fact["Date"], format="%Y-%m-%d", errors="coerce").isna()).sum()
n_amt_fmt = (fact["Amount, $"].notna() & pd.to_numeric(fact["Amount, $"], errors="coerce").isna()).sum()

def n_variant(col, canon):
    """Số ô có nhãn khác nhãn chuẩn (không tính ô thiếu)."""
    return (fact[col].notna() & ~fact[col].isin(canon)).sum()

# Calendar: số ngày thiếu so với 365 ngày của năm 2023
cal_days = pd.to_datetime(tables["Calendar"]["Date"])
n_cal = len(pd.date_range("2023-01-01", "2023-12-31").difference(cal_days))

rows = [
    # (Nhóm lỗi, Vấn đề, Cột, Số dòng, Nhận định, Hướng xử lý)
    ("Thiếu", "Channel thiếu", "Channel", fact["Channel"].isna().sum(),
     "Có liên hệ với dòng sản phẩm; gợi ý MAR",
     "Xem xét điền mode theo dòng sản phẩm hoặc giữ Unknown"),
    ("Thiếu", "Region thiếu", "Region", fact["Region"].isna().sum(),
     "Chưa phát hiện liên hệ với các yếu tố kiểm tra",
     "Xem xét điền mode hoặc giữ Unknown"),
    ("Thiếu", "Amount thiếu", "Amount, $", fact["Amount, $"].isna().sum(),
     "Có liên hệ với dòng sản phẩm, nhóm tài chính và quy mô Subgroup",
     "Thử điền trung vị theo Subgroup; so sánh với phương án loại dòng"),
    ("Thiếu", "Date thiếu", "Date", fact["Date"].isna().sum(),
     "Chưa phát hiện liên hệ với dòng sản phẩm và nhóm tài chính; chưa xác định cơ chế thiếu",
     "Cân nhắc loại khi phân tích theo tháng"),
    ("Bất thường", "Amount ngoại lai", "Amount, $", d["Outlier"].sum(),
     "Giá trị bất thường so với Subgroup",
     "Kiểm tra và đánh giá tác động trước khi xử lý"),
    ("Bất thường", "Dấu Amount trái quy ước", "Amount, $", sign_bad.sum(),
     "Dấu không phù hợp với nhóm tài chính",
     "Xác minh nhóm tài chính trước khi sửa dấu"),
    ("Không nhất quán", "Date nhiều định dạng", "Date", n_date_fmt,
     "4 kiểu định dạng", "Parse từng định dạng, chuẩn hóa về YYYY-MM-DD"),
    ("Không nhất quán", "Amount nhiều định dạng", "Amount, $", n_amt_fmt,
     "5 kiểu chuỗi ($, ngoặc, USD, kiểu VN...)", "Dùng hàm parse_amount, đổi sang số"),
    ("Không nhất quán", "Nhãn Channel viết khác nhau", "Channel", n_variant("Channel", ["Online", "Store", "Distributor"]),
     "Khác hoa thường, thừa khoảng trắng", "strip, lower rồi map về nhãn chuẩn"),
    ("Không nhất quán", "Nhãn Region viết khác nhau", "Region", n_variant("Region", ["Bắc", "Trung", "Nam"]),
     "Không dấu, thêm 'Miền', hoa thường", "strip, lower rồi map về nhãn chuẩn"),
    ("Không nhất quán", "Nhãn Revenue/Expense viết khác nhau", "Revenue or expense", n_variant("Revenue or expense", ["Revenue", "Expense"]),
     "Viết tắt, hoa thường", "strip, lower rồi map về nhãn chuẩn"),
    ("Không nhất quán", "Revenue/Expense trái với nhóm", "Revenue or expense", re_bad.sum(),
     "Mâu thuẫn với Subgroup", "Suy lại nhãn từ Subgroup ID"),
    ("Không nhất quán", "Group ID không khớp Subgroup", "Group ID", grp_bad.sum(),
     "Mâu thuẫn với bảng Subgroup", "Gán lại Group ID từ Subgroup ID"),
    ("Không nhất quán", "Business Line ID không khớp tên", "Business Line ID", bl_bad.sum(),
     "ID và tên dòng sản phẩm mâu thuẫn",
     "Đối chiếu bảng danh mục, xác minh trước khi sửa ID hoặc tên"),
    ("Trùng lặp", "Dòng trùng hoàn toàn", "Toàn bộ dòng", fact.duplicated().sum(),
     "Trùng cả TransactionID và nội dung", "drop_duplicates"),
    ("Tích hợp", "Subgroup ID không có trong danh mục", "Subgroup ID", invalid_keys["Subgroup ID"],
     "Khóa mồ côi", "Kiểm tra nguồn; loại hoặc gán 'Unknown'; không inner join"),
    ("Tích hợp", "Business Line ID không có trong danh mục", "Business Line ID", invalid_keys["Business Line ID"],
     "Khóa mồ côi (ID = 4)", "Suy lại từ Business Line Name"),
]
diag = pd.DataFrame(rows, columns=["Nhóm lỗi", "Vấn đề", "Cột", "Số dòng", "Nhận định", "Hướng xử lý"])
diag["Tỷ lệ %"] = (diag["Số dòng"] / n * 100).round(2)

# Calendar tính theo ngày (không phải dòng của Fact) nên thêm riêng, không có tỷ lệ %
diag.loc[len(diag)] = ["Tích hợp", "Calendar thiếu ngày", "Calendar", n_cal,
                       f"Thiếu {n_cal}/365 ngày trong bảng ngày", "Dựng lại từ date_range; dùng left join", np.nan]
diag

,Nhóm lỗi,Vấn đề,Cột,Số dòng,Nhận định,Hướng xử lý,Tỷ lệ %
0,Thiếu,Channel thiếu,Channel,819,Có liên hệ với dòng sản phẩm; gợi ý MAR,Xem xét điền mode theo dòng sản phẩm hoặc giữ ...,9.39
1,Thiếu,Region thiếu,Region,452,Chưa phát hiện liên hệ với các yếu tố kiểm tra,Xem xét điền mode hoặc giữ Unknown,5.18
2,Thiếu,Amount thiếu,"Amount, $",253,"Có liên hệ với dòng sản phẩm, nhóm tài chính v...",Thử điền trung vị theo Subgroup; so sánh với p...,2.90
3,Thiếu,Date thiếu,Date,101,Chưa phát hiện liên hệ với dòng sản phẩm và nh...,Cân nhắc loại khi phân tích theo tháng,1.16
4,Bất thường,Amount ngoại lai,"Amount, $",72,Giá trị bất thường so với Subgroup,Kiểm tra và đánh giá tác động trước khi xử lý,0.83
5,Bất thường,Dấu Amount trái quy ước,"Amount, $",87,Dấu không phù hợp với nhóm tài chính,Xác minh nhóm tài chính trước khi sửa dấu,1.00
6,Không nhất quán,Date nhiều định dạng,Date,2628,4 kiểu định dạng,"Parse từng định dạng, chuẩn hóa về YYYY-MM-DD",30.12
7,Không nhất quán,Amount nhiều định dạng,"Amount, $",6779,"5 kiểu chuỗi ($, ngoặc, USD, kiểu VN...)","Dùng hàm parse_amount, đổi sang số",77.69
8,Không nhất quán,Nhãn Channel viết khác nhau,Channel,436,"Khác hoa thường, thừa khoảng trắng","strip, lower rồi map về nhãn chuẩn",5.00
9,Không nhất quán,Nhãn Region viết khác nhau,Region,439,"Không dấu, thêm 'Miền', hoa thường","strip, lower rồi map về nhãn chuẩn",5.03
